In [1]:
import tensorflow as tf

In [2]:
from tensorflow.keras.layers import Input, Reshape, Dropout, Dense

In [3]:
from tensorflow.keras.layers import Flatten, BatchNormalization

In [4]:
from tensorflow.keras.layers import Activation, ZeroPadding2D

In [5]:
from tensorflow.keras.layers import LeakyReLU

In [6]:
from tensorflow.keras.layers import UpSampling2D, Conv2D

In [7]:
from tensorflow.keras.models import Sequential, Model, load_model

In [8]:
from tensorflow.keras.optimizers import Adam

In [10]:
import zipfile, numpy as np, os, time
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt


In [12]:
def time_string(sec_elapsed):
     hour = int(sec_elapsed / (60 * 60))
     minute = int((sec_elapsed % (60 * 60)) / 60)
     second = sec_elapsed % 60
     return "{}:{:>02}:{:>05.2f}".format(hour, minute, second)

In [13]:
gen_res = 3
gen_square = 32 * gen_res
img_chan = 3
img_rows = 5
img_cols = 5
img_margin = 16
seed_vector = 200

In [26]:
data_path = 'C:/Users/Administrator/Downloads/fruits360/fruits360_filtered/Training'

In [15]:
epochs = 5000
num_batch = 32
num_buffer = 60000

In [16]:
training_binary_path = os.path.join(data_path, f'training_data_{gen_square}_{gen_square}.npy')

In [ ]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import tensorflow as tf
from tensorflow.keras.layers import Input, Reshape, Dropout, Dense
from tensorflow.keras.layers import Flatten, BatchNormalization
from tensorflow.keras.layers import Activation, ZeroPadding2D
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.layers import UpSampling2D, Conv2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

import numpy as np
from PIL import Image
from tqdm import tqdm
import os, time
import matplotlib.pyplot as plt


# ==============================
# 2. TIME FUNCTION
# ==============================
def time_string(sec_elapsed):
    hour = int(sec_elapsed / (60 * 60))
    minute = int((sec_elapsed % (60 * 60)) / 60)
    second = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(hour, minute, second)


# ==============================
# 3. PARAMETERS
# ==============================
gen_res = 3
gen_square = 32 * gen_res   # 96x96 images
img_chan = 3

img_rows = 5
img_cols = 5
img_margin = 16

seed_vector = 200

data_path = 'C:/Users/Administrator/Downloads/fruits360/fruits360_filtered/Training'
epochs = 5000
num_batch = 32
num_buffer = 60000

print(f"Image size: {gen_square}x{gen_square}")


# ==============================
# 4. LOAD & PREPROCESS IMAGES
# ==============================
training_binary_path = os.path.join(
    data_path, f'training_data_{gen_square}_{gen_square}.npy'
)

if not os.path.isfile(training_binary_path):
    print("Processing images...")
    train_data = []

    images_path = os.path.join(data_path, 'tomato 1')  # ⚠️ ensure exists

    for filename in tqdm(os.listdir(images_path)):
        path = os.path.join(images_path, filename)

        image = Image.open(path).resize((gen_square, gen_square))
        image = np.asarray(image)

        if image.shape == (gen_square, gen_square, 3):
            train_data.append(image)

    train_data = np.array(train_data).astype(np.float32)
    train_data = train_data / 127.5 - 1.0   # normalize [-1,1]

    np.save(training_binary_path, train_data)
else:
    print("Loading saved dataset...")
    train_data = np.load(training_binary_path)


# ==============================
# 5. DATASET
# ==============================
train_dataset = tf.data.Dataset.from_tensor_slices(train_data)\
                    .shuffle(num_buffer).batch(num_batch)


# ==============================
# 6. GENERATOR
# ==============================
def create_generator(seed_size, channels):
    model = Sequential()

    model.add(Input(shape=(seed_size,)))

    model.add(Dense(4*4*256, activation="relu"))
    model.add(Reshape((4, 4, 256)))

    model.add(UpSampling2D())
    model.add(Conv2D(256, 3, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Activation("relu"))

    model.add(UpSampling2D())
    model.add(Conv2D(256, 3, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Activation("relu"))

    model.add(UpSampling2D())
    model.add(Conv2D(128, 3, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Activation("relu"))

    if gen_res > 1:
        model.add(UpSampling2D(size=(gen_res, gen_res)))
        model.add(Conv2D(128, 3, padding="same"))
        model.add(BatchNormalization(momentum=0.8))
        model.add(Activation("relu"))

    model.add(Conv2D(channels, 3, padding="same"))
    model.add(Activation("tanh"))

    return model


# ==============================
# 7. DISCRIMINATOR
# ==============================
def create_discriminator(image_shape):
    model = Sequential()

    model.add(Input(shape=image_shape))

    model.add(Conv2D(32, 3, strides=2, padding="same"))
    model.add(LeakyReLU(0.2))
    model.add(Dropout(0.25))

    model.add(Conv2D(64, 3, strides=2, padding="same"))
    model.add(ZeroPadding2D(((0,1),(0,1))))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(0.2))
    model.add(Dropout(0.25))

    model.add(Conv2D(128, 3, strides=2, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(0.2))
    model.add(Dropout(0.25))

    model.add(Conv2D(256, 3, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(0.2))
    model.add(Dropout(0.25))

    model.add(Conv2D(512, 3, padding="same"))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(0.2))
    model.add(Dropout(0.25))

    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))

    return model


# ==============================
# 8. INITIALIZE MODELS
# ==============================
generator = create_generator(seed_vector, img_chan)
img_shape = (gen_square, gen_square, img_chan)
discriminator = create_discriminator(img_shape)

disc_optimizer = Adam(1.5e-4, 0.5)
gen_optimizer = Adam(1.5e-4, 0.5)

discriminator.compile(loss='binary_crossentropy', optimizer=disc_optimizer)


# ==============================
# 9. LOSS FUNCTIONS
# ==============================
cross_entropy = tf.keras.losses.BinaryCrossentropy()

def discrim_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def gen_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)


# ==============================
# 10. TRAIN STEP
# ==============================
@tf.function
def train_step(images):
    seed = tf.random.normal([num_batch, seed_vector])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_imgs = generator(seed, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(gen_imgs, training=True)

        g_loss = gen_loss(fake_output)
        d_loss = discrim_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(
        g_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(
        d_loss, discriminator.trainable_variables)

    gen_optimizer.apply_gradients(zip(
        gradients_of_generator, generator.trainable_variables))

    disc_optimizer.apply_gradients(zip(
        gradients_of_discriminator, discriminator.trainable_variables))

    return g_loss, d_loss


# ==============================
# 11. SAVE IMAGES
# ==============================
def save_images(cnt, noise):
    img_array = np.full((img_margin + (img_rows*(gen_square+img_margin)),
                         img_margin + (img_cols*(gen_square+img_margin)), 3),
                         255, dtype=np.uint8)

    gen_imgs = generator.predict(noise)
    gen_imgs = 0.5 * gen_imgs + 0.5

    img_count = 0
    for row in range(img_rows):
        for col in range(img_cols):
            r = row*(gen_square+16)+img_margin
            c = col*(gen_square+16)+img_margin
            img_array[r:r+gen_square,c:c+gen_square] = gen_imgs[img_count]*255
            img_count += 1

    output_path = os.path.join(data_path,'output')
    os.makedirs(output_path, exist_ok=True)

    Image.fromarray(img_array).save(
        os.path.join(output_path,f"train-{cnt}.png")
    )


# ==============================
# 12. TRAIN LOOP
# ==============================
def train(dataset, epochs):
    fixed_seed = np.random.normal(0,1,(img_rows*img_cols,seed_vector))

    for epoch in range(epochs):
        g_losses, d_losses = [], []

        for image_batch in dataset:
            g_loss, d_loss = train_step(image_batch)
            g_losses.append(g_loss)
            d_losses.append(d_loss)

        print(f"Epoch {epoch+1}, G={np.mean(g_losses)}, D={np.mean(d_losses)}")

        if epoch % 50 == 0:
            save_images(epoch, fixed_seed)


# ==============================
# 13. START TRAINING
# ==============================
train(train_dataset, epochs)


# ==============================
# 14. VIEW RESULTS
# ==============================
img = imread(os.path.join(data_path,'output/train-100.png'))
plt.imshow(img)
plt.axis('off')

Image size: 96x96
Loading saved dataset...
